# 08 — Modelagem

**Objetivo:** treinar e comparar modelos de classificação com validação temporal.

**Modelos:**
* Regressão Logística (baseline interpretável — coeficientes como WOE-like)
* Árvore de Decisão
* Random Forest
* Gradient Boosted Trees

**Protocolo:**
* Treino nos snapshots 06–12/05, seleção pela **validação** (13–14/05)
* `StandardScaler` dentro do `Pipeline` — ajustado apenas no treino (anti-leakage)
* Métricas na validação: **AUC-ROC** e **KS**
* Todos os experimentos registrados no **MLflow**
* Previsões de validação e teste salvas em `model_predictions` para o `09_avaliacao`

> 💡 Para ir além: LightGBM/XGBoost via `synapse.ml.lightgbm` e tuning com Optuna (`pip install optuna`) são extensões naturais deste notebook.

In [0]:
# ── Configuração central (idem notebook 00) ──
CATALOG = "workspace"
SCHEMA  = "ml_bakehouse"

import mlflow
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import (LogisticRegression, DecisionTreeClassifier,
                                       RandomForestClassifier, GBTClassifier)
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

train_df = spark.table(f"{CATALOG}.{SCHEMA}.train_base")
val_df   = spark.table(f"{CATALOG}.{SCHEMA}.validation_base")
test_df  = spark.table(f"{CATALOG}.{SCHEMA}.test_base")

feat_cols = [c for c in train_df.columns if c not in ("customerID", "reference_date", "target")]
print("Features:", feat_cols)

def ks_statistic(pdf, score_col="score", label_col="target"):
    """KS = máxima separação entre CDF de eventos e não eventos."""
    pdf = pdf.sort_values(score_col, ascending=False).reset_index(drop=True)
    cum_ev  = pdf[label_col].cumsum() / pdf[label_col].sum()
    cum_nev = (1 - pdf[label_col]).cumsum() / (1 - pdf[label_col]).sum()
    return float((cum_ev - cum_nev).abs().max())

In [0]:
# Loop de modelos: treino → AUC/KS na validação → registro MLflow
assembler = VectorAssembler(inputCols=feat_cols, outputCol="raw_features", handleInvalid="keep")
scaler    = StandardScaler(inputCol="raw_features", outputCol="features")
evaluator = BinaryClassificationEvaluator(labelCol="target", metricName="areaUnderROC")

modelos = {
    "regressao_logistica": LogisticRegression(featuresCol="features", labelCol="target", maxIter=100),
    "arvore_decisao":      DecisionTreeClassifier(featuresCol="features", labelCol="target", seed=42),
    "random_forest":       RandomForestClassifier(featuresCol="features", labelCol="target", numTrees=200, seed=42),
    "gbt":                 GBTClassifier(featuresCol="features", labelCol="target", maxIter=50, seed=42),
}

resultados, preds_all = [], None
for nome, algo in modelos.items():
    pipe = Pipeline(stages=[assembler, scaler, algo])
    with mlflow.start_run(run_name=nome):
        model = pipe.fit(train_df)

        val_preds = model.transform(val_df).withColumn("score", vector_to_array("probability")[1])
        auc_val = evaluator.evaluate(val_preds)
        ks_val  = ks_statistic(val_preds.select("target", "score").toPandas())

        mlflow.log_param("modelo", nome)
        mlflow.log_param("n_features", len(feat_cols))
        mlflow.log_metric("val_auc", round(auc_val, 4))
        mlflow.log_metric("val_ks", round(ks_val, 4))
        mlflow.spark.log_model(model, nome)

        # Previsões de validação e teste para o 09_avaliacao
        test_preds = model.transform(test_df).withColumn("score", vector_to_array("probability")[1])
        for base, tag in [(val_preds, "validacao"), (test_preds, "teste")]:
            p = base.select(
                F.lit(nome).alias("modelo"), F.lit(tag).alias("base"),
                "reference_date", "customerID", "target",
                F.col("score").cast("double").alias("score"),
            )
            preds_all = p if preds_all is None else preds_all.unionByName(p)

        resultados.append((nome, round(auc_val, 4), round(ks_val, 4)))

print("Treinamento concluído.")
display(spark.createDataFrame(resultados, "modelo string, val_auc double, val_ks double")
        .orderBy(F.desc("val_auc")))

In [0]:
# Gravação das previsões (validação + teste, todos os modelos)
(preds_all.write.mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable(f"{CATALOG}.{SCHEMA}.model_predictions"))

print(f"Salvo: {CATALOG}.{SCHEMA}.model_predictions}", preds_all.count(), "linhas")
display(spark.table(f"{CATALOG}.{SCHEMA}.model_predictions").limit(10))

## Interpretação e próximos passos

* A tabela exibida ordena os modelos por **AUC na validação** — critério primário de seleção (com KS como visão complementar de separação).
* Consulte os experimentos no MLflow: ícone **Experiments** na barra lateral (ou Workspace → Experiments) para ver runs, parâmetros, métricas e modelos registrados.
* Expectativa realista: dataset sintético → AUC próximo de 0,5–0,6. O valor pedagógico está no **processo** correto, não em bater recordes de métrica.
* **Extensões:** `CrossValidator` + `ParamGridBuilder` para tuning; `synapse.ml.lightgbm.LightGBMClassifier` como candidato mais forte; Optuna para busca bayesiana.
* **Próximo passo:** `09_avaliacao` — métricas completas no teste, lift, calibração e estabilidade.